# 06 — Create Addresses

For every subscription with a successfully resolved address (from
`05_Fetch_Subscriptions.ipynb`'s Voyager lookup), `PUT`s that address onto
its target account:

```
{{host}}/rest/SubscriberService/v1/subscribers/{accountcode}
```

`addLine1` / `addLine2` / `city` / `zip` / `state` (region ISO) all come
from the Voyager `ParsedAddress_*` columns — real values, not placeholders.

Returns each new address's `id`, saved as `ship_add_id` — this is what
`07_Create_Subscription_Orders.ipynb` uses as `shipAddId` on the order.

Subscriptions whose Voyager lookup didn't resolve are skipped here (flagged
with status `"skipped"`) — they need a manual address before their order can
be created.

## 1. Setup

In [21]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_addresses")

df_subscriptions = load_subscriptions_resolved()
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from 05_Fetch_Subscriptions.ipynb")


2026-07-23 10:03:00,866 [INFO] Loaded 323 subscriptions from 05_Fetch_Subscriptions.ipynb


## 2. Per-subscription address creation

In [22]:
def create_address_for_subscription(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  account_number,
        "addLine1":             row.get("ParsedAddress_addLine1"),
        "status":               "failed",
        "ship_add_id":          None,
        "error":                None,
    }

    if not row.get("ParsedAddress_parsed_ok"):
        result["status"] = "skipped"
        result["error"] = row.get("ParsedAddress_error") or "Voyager address lookup failed — needs manual address"
        return result

    status, ship_add_id, error = add_address_to_account(
        session,
        account_number,
        row["ParsedAddress_addLine1"],
        location_id=row.get("ParsedAddress_location_id") or str(subscription_id),
        address2=row.get("ParsedAddress_addLine2"),
        city=row.get("ParsedAddress_city") or "Christchurch",
        zip_code=row.get("ParsedAddress_postcode") or "1234",
        region_iso=row.get("ParsedAddress_region_iso"),
    )
    result["status"] = status
    result["ship_add_id"] = ship_add_id
    result["error"] = error

    if status == "created":
        logger.info(f"[OK] subscription {subscription_id} -> {account_number} address id={ship_add_id}")
    else:
        logger.error(f"[FAIL] subscription {subscription_id} — {error}")

    return result


## 3. Run (parallel driver)

In [23]:
def create_all_addresses(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []

    logger.info(f"Creating addresses for {total:,} subscriptions with {max_workers} workers...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_address_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} — {ok} created so far")

    return pd.DataFrame(results)


df_address_results = create_all_addresses(df_subscriptions)
df_address_results.head(20)


2026-07-23 10:03:00,984 [INFO] Creating addresses for 323 subscriptions with 10 workers...
2026-07-23 10:03:19,573 [ERROR] [FAIL] subscription V113062418 — None
2026-07-23 10:03:19,608 [ERROR] [FAIL] subscription V113062400 — None
2026-07-23 10:03:19,748 [ERROR] [FAIL] subscription V113062962 — None
2026-07-23 10:03:19,774 [ERROR] [FAIL] subscription V113061451 — None
2026-07-23 10:03:21,075 [ERROR] [FAIL] subscription V113062988 — None
2026-07-23 10:03:21,163 [ERROR] [FAIL] subscription V113062392 — None
2026-07-23 10:03:21,181 [ERROR] [FAIL] subscription V113062970 — None
2026-07-23 10:03:21,214 [ERROR] [FAIL] subscription V113062616 — None
2026-07-23 10:03:21,398 [ERROR] [FAIL] subscription V113062384 — None
2026-07-23 10:03:21,439 [ERROR] [FAIL] subscription V113062632 — None
2026-07-23 10:03:30,995 [ERROR] [FAIL] subscription V113062996 — None
2026-07-23 10:03:31,007 [ERROR] [FAIL] subscription V113063002 — None
2026-07-23 10:03:31,517 [ERROR] [FAIL] subscription V113063036 — None

,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
0,V113062897,999656921_fullbatch,NaN,skipped,None,circuits lookup failed: 404 Client Error: Not ...
1,V113062905,99965692_fullbatch,NaN,skipped,None,circuits lookup failed: 404 Client Error: Not ...
2,V113062335,99965692_fullbatch,NaN,skipped,None,no SupplierServiceID on this subscription
3,V113062343,99965692_fullbatch,NaN,skipped,None,no SupplierServiceID on this subscription
4,V113062913,99965692_fullbatch,NaN,skipped,None,circuits lookup failed: 404 Client Error: Not ...
5,V113062418,99965692_fullbatch,11 Matata Way,exists,141207,None
6,V113062400,99965692_fullbatch,5 Matata Way,exists,141205,None
7,V113062962,99965692_fullbatch,18/10 Fathom Place,exists,141206,None
8,V113061451,99965692_fullbatch,39 Chester Street West,exists,141105,None
9,V113062988,99965692_fullbatch,3/23 Awaroa Road,exists,141108,None


## 4. Failures / skips

In [24]:
not_created = df_address_results[df_address_results["status"] != "created"]
print(f"{len(not_created):,} / {len(df_address_results):,} addresses not created (failed or skipped)")
not_created.groupby("status").size()


193 / 323 addresses not created (failed or skipped)


status
exists     126
skipped     67
dtype: int64

## 5. Save

In [25]:
save_df("address_results", df_address_results)


Saved 323 rows -> migration_data\06_address_creation_results.csv
